In [ ]:
print("Automated CI/CD Pipeline Risk and Failure Classification")

In [ ]:
import os,hashlib,warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

In [ ]:
QUICK=False
RANDOM_STATE=42
print(QUICK)

In [ ]:
BASE=".." if os.path.basename(os.getcwd())=="notebooks" else "."
DATA_DIR,MODEL_DIR=f"{BASE}/data",f"{BASE}/models"
GADFPD_CSV=f"{DATA_DIR}/final_research_dataset_MASTER.csv"
OWN_JSONL=f"{DATA_DIR}/runs_own.jsonl"
os.makedirs(MODEL_DIR,exist_ok=True)
print(os.path.exists(GADFPD_CSV),os.path.exists(OWN_JSONL))

In [ ]:
raw=pd.read_csv(GADFPD_CSV,low_memory=False)

In [ ]:
raw.shape

In [ ]:
print(raw['conclusion'].value_counts(dropna=False))

In [ ]:
print(raw['status'].value_counts(dropna=False))

In [ ]:
OUTCOMES={"success":0,"passed":0,"failure":1,"failed":1,"errored":1}
raw['outcome']=raw['conclusion'].where(raw['conclusion'].notna(),raw['status'])

In [ ]:
raw['label']=raw['outcome'].map(OUTCOMES)

In [ ]:
print(raw['label'].value_counts(dropna=False))

In [ ]:
g=raw[raw['label'].notna()].copy()
print(g.shape)

In [ ]:
g=g[g['run_attempt'].isna()|(g['run_attempt']==1)]
print(g.shape)

In [ ]:
g=g.drop_duplicates('run_id')
print(g.shape)

In [ ]:
g['label']=g['label'].astype(int)

In [ ]:
g['time']=pd.to_datetime(g['timestamp_raw'],errors='coerce',utc=True)
print(g['time'].isnull().sum())

In [ ]:
g['branch_is_main']=g['head_branch'].isin(['main','master']).astype(float).where(g['head_branch'].notna())

In [ ]:
g['is_merge']=g['is_merge'].astype(float)

In [ ]:
SALT=os.environ.get("HASH_SALT","ned-ct378")
g['repo_id']=g['repo'].map(lambda x:hashlib.sha256((SALT+str(x)).encode()).hexdigest()[:12])
print(g['repo_id'].nunique())

In [ ]:
g['source']="GADFPD"

In [ ]:
KEEP=["run_id","repo_id","workflow_id","time","event","branch_is_main","additions","deletions","files_modified","msg_len","is_merge","num_parents","time_since_last_commit","run_number","label","source"]
g=g[KEEP]
print(g.shape)

In [ ]:
own=pd.read_json(OWN_JSONL,lines=True)
print(own.shape)

In [ ]:
print(own['label'].value_counts())

In [ ]:
own['time']=pd.to_datetime(own['created_at'],utc=True)

In [ ]:
own['repo_id']=own['repo']
own['source']="own_API"

In [ ]:
own=own[[c for c in KEEP if c in own.columns]]
print(own.shape)

In [ ]:
data=pd.concat([g,own],ignore_index=True).drop_duplicates('run_id')
print(data.shape)

In [ ]:
data=data.dropna(subset=['time']).sort_values('time').reset_index(drop=True)
print(data.shape)

In [ ]:
print(data['source'].value_counts())

In [ ]:
print(data['label'].value_counts())

In [ ]:
print(data.isnull().sum())

In [ ]:
data['churn_total']=data['additions']+data['deletions']
data['churn_per_file']=data['churn_total']/data['files_modified'].clip(lower=1)

In [ ]:
data['hour']=data['time'].dt.hour
data['dow']=data['time'].dt.dayofweek

In [ ]:
by_repo=data.groupby('repo_id')['label']
data['repo_prev']=by_repo.shift(1)

In [ ]:
data['repo_rate10']=by_repo.transform(lambda s:s.shift(1).rolling(10,min_periods=1).mean())

In [ ]:
data['repo_prior_runs']=data.groupby('repo_id').cumcount()

In [ ]:
has_wf=data['workflow_id'].notna()
data['wf_key']=data['repo_id']+"_"+data['workflow_id'].fillna(-1).astype(int).astype(str)
print(has_wf.sum())

In [ ]:
by_wf=data.groupby('wf_key')['label']
data['wf_prev']=by_wf.shift(1).where(has_wf)
data['wf_rate10']=by_wf.transform(lambda s:s.shift(1).rolling(10,min_periods=1).mean()).where(has_wf)

In [ ]:
print(data[['repo_prev','repo_rate10','repo_prior_runs','wf_prev','wf_rate10']].describe())

In [ ]:
LOG_COLS=["additions","deletions","files_modified","msg_len","churn_total","churn_per_file","time_since_last_commit"]
NUM_COLS=["num_parents","is_merge","run_number","hour","dow","branch_is_main","repo_prev","repo_rate10","repo_prior_runs","wf_prev","wf_rate10"]
CAT_COLS=["event"]
FEATURES=LOG_COLS+NUM_COLS+CAT_COLS
print(len(FEATURES),FEATURES)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
print(data[LOG_COLS+NUM_COLS].describe().T[["count","mean","50%","max"]].round(2))

In [ ]:
cb=data['label'].value_counts().sort_index()
plt.figure(figsize=(5,4))
plt.bar(["success","failure"],cb.values,color=["#4c9f70","#d1495b"])
for i,v in enumerate(cb.values): plt.text(i,v,f"{v:,}\n({v/len(data):.1%})",ha="center",va="bottom")
plt.title("Class balance")
plt.show()

In [ ]:
miss=data[FEATURES].isna().mean().sort_values(ascending=False).head(12)
plt.figure(figsize=(6,4))
miss.iloc[::-1].plot.barh(color="#6c757d")
plt.title("Missing values (top 12)")
plt.xlabel("fraction missing")
plt.tight_layout()
plt.show()

In [ ]:
ev=data.groupby(data['event'].fillna("unknown"))['label'].agg(['mean','size']).sort_values('size',ascending=False).head(6)
plt.figure(figsize=(6,4))
plt.bar(ev.index,ev['mean'],color="#edae49")
plt.xticks(rotation=30)
plt.title("Failure rate by trigger event")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(data=data.assign(log_churn=np.log1p(data['churn_total'].clip(lower=0)),cls=data['label'].map({0:"success",1:"failure"})),x="log_churn",hue="cls",stat="density",common_norm=False,bins=40,palette={"success":"#4c9f70","failure":"#d1495b"})
plt.title("log(1 + churn) by class")
plt.show()

In [ ]:
m=data[data['time']>="2025-01-01"].set_index('time')['label'].resample("MS").agg(['mean','size'])
plt.figure(figsize=(7,4))
plt.plot(m.index,m['mean'],marker="o",color="#d1495b")
plt.title("Monthly failure rate (2025 onward)")
plt.ylabel("failure rate")
plt.show()

In [ ]:
corr=data[LOG_COLS+NUM_COLS+['label']].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr,cmap="coolwarm",center=0,vmin=-1,vmax=1,square=True,linewidths=0.5)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5,4))
sns.boxplot(data=data.assign(log_files=np.log1p(data['files_modified'].clip(lower=0)),cls=data['label'].map({0:"success",1:"failure"})),x="cls",y="log_files",palette={"success":"#4c9f70","failure":"#d1495b"})
plt.title("Outliers: log(1 + files modified) by class")
plt.show()

In [ ]:
rr=data.groupby('repo_id')['label'].agg(['mean','size'])
rr=rr[rr['size']>=200].sort_values('mean',ascending=False).head(15)
plt.figure(figsize=(8,4.5))
plt.barh(rr.index[::-1],rr['mean'][::-1],color="#00798c")
plt.title("Failure rate: 15 repos with most failures (>=200 runs)")
plt.xlabel("failure rate")
plt.tight_layout()
plt.show()

In [ ]:
n=len(data)
i1,i2=int(n*0.70),int(n*0.85)
train,val,test=data.iloc[:i1],data.iloc[i1:i2],data.iloc[i2:]

In [ ]:
X_tr,y_tr=train[FEATURES],train['label']
X_va,y_va=val[FEATURES],val['label']
X_te,y_te=test[FEATURES],test['label']

In [ ]:
for name,d in [("train",train),("val",val),("test",test)]:
    print(f"{name:5s} {len(d):7d} rows  {d['time'].min():%Y-%m-%d} -> {d['time'].max():%Y-%m-%d}  failure rate {d['label'].mean():.3f}")

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,FunctionTransformer

In [ ]:
import sys
sys.path.append(BASE)
from ci_utils import log1p_clip
print(log1p_clip.__module__)

In [ ]:
prep=ColumnTransformer([
    ("log",Pipeline([("imp",SimpleImputer(strategy="median")),("log",FunctionTransformer(log1p_clip)),("sc",StandardScaler())]),LOG_COLS),
    ("num",Pipeline([("imp",SimpleImputer(strategy="median",add_indicator=True)),("sc",StandardScaler())]),NUM_COLS),
    ("cat",Pipeline([("imp",SimpleImputer(strategy="constant",fill_value="unknown")),("oh",OneHotEncoder(handle_unknown="ignore"))]),CAT_COLS)])
print(prep)

In [ ]:
Xt=clone(prep).fit(X_tr).transform(X_va)
print(Xt.shape)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score,precision_score,recall_score,roc_auc_score,average_precision_score
dummy=Pipeline([("prep",clone(prep)),("clf",DummyClassifier(strategy="most_frequent"))]).fit(X_tr,y_tr)
p_dummy_val=dummy.predict_proba(X_va)[:,1]
print(p_dummy_val[:5])

In [ ]:
rule_val=val['wf_prev'].fillna(val['repo_prev']).fillna(0).values
print(rule_val[:5])

In [ ]:
base_rows=[]
for name,proba in [("Baseline: majority class",p_dummy_val),("Baseline: previous-run outcome",rule_val)]:
    pred=(proba>=0.5).astype(int)
    base_rows.append({"Model":name,"macro_F1":f1_score(y_va,pred,average="macro"),"fail_F1":f1_score(y_va,pred,pos_label=1),"fail_prec":precision_score(y_va,pred,zero_division=0),"fail_rec":recall_score(y_va,pred),"ROC_AUC":roc_auc_score(y_va,proba),"PR_AUC":average_precision_score(y_va,proba)})
base_df=pd.DataFrame(base_rows).set_index("Model").round(3)
print(base_df)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV,TimeSeriesSplit

In [ ]:
TUNE_ROWS=20000 if QUICK else 80000
N_ITER=2 if QUICK else 8
N_TREES=60 if QUICK else 200
print(TUNE_ROWS,N_ITER,N_TREES)
print("Tuning uses the most recent",min(TUNE_ROWS,len(X_tr)),"of",len(X_tr),"training rows (time order kept). Final models are refit on all",len(X_tr),"training rows.")

In [ ]:
cv=TimeSeriesSplit(n_splits=5)
X_sub,y_sub=X_tr.iloc[-TUNE_ROWS:],y_tr.iloc[-TUNE_ROWS:]
fitted={}
print(X_sub.shape)

In [ ]:
lr_grid=RandomizedSearchCV(Pipeline([("prep",clone(prep)),("clf",LogisticRegression(max_iter=500,class_weight="balanced"))]),{"clf__C":[0.01,0.1,1,10]},n_iter=min(N_ITER,4),scoring="f1_macro",cv=cv,random_state=RANDOM_STATE,n_jobs=1,refit=False)
lr_grid.fit(X_sub,y_sub)
print("Logistic Regression:",lr_grid.best_params_,lr_grid.best_score_)

In [ ]:
fitted["Logistic Regression"]=Pipeline([("prep",clone(prep)),("clf",LogisticRegression(max_iter=500,class_weight="balanced",C=lr_grid.best_params_["clf__C"]))]).fit(X_tr,y_tr)

In [ ]:
rf_grid=RandomizedSearchCV(Pipeline([("prep",clone(prep)),("clf",RandomForestClassifier(n_estimators=N_TREES,class_weight="balanced_subsample",n_jobs=-1,random_state=RANDOM_STATE))]),{"clf__max_depth":[8,12,16,None],"clf__min_samples_leaf":[5,20,50]},n_iter=min(N_ITER,12),scoring="f1_macro",cv=cv,random_state=RANDOM_STATE,n_jobs=1,refit=False)
rf_grid.fit(X_sub,y_sub)
print("Random Forest:",rf_grid.best_params_,rf_grid.best_score_)

In [ ]:
fitted["Random Forest"]=Pipeline([("prep",clone(prep)),("clf",RandomForestClassifier(n_estimators=N_TREES,class_weight="balanced_subsample",n_jobs=-1,random_state=RANDOM_STATE))]).set_params(**rf_grid.best_params_).fit(X_tr,y_tr)

In [ ]:
hgb_grid=RandomizedSearchCV(Pipeline([("prep",clone(prep)),("clf",HistGradientBoostingClassifier(class_weight="balanced",random_state=RANDOM_STATE))]),{"clf__learning_rate":[0.03,0.1,0.2],"clf__max_leaf_nodes":[15,31,63],"clf__min_samples_leaf":[20,50,100]},n_iter=min(N_ITER,27),scoring="f1_macro",cv=cv,random_state=RANDOM_STATE,n_jobs=1,refit=False)
hgb_grid.fit(X_sub,y_sub)
print("Hist Gradient Boosting:",hgb_grid.best_params_,hgb_grid.best_score_)

In [ ]:
fitted["Hist Gradient Boosting"]=Pipeline([("prep",clone(prep)),("clf",HistGradientBoostingClassifier(class_weight="balanced",random_state=RANDOM_STATE))]).set_params(**hgb_grid.best_params_).fit(X_tr,y_tr)
print(list(fitted))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

In [ ]:
dt_grid=RandomizedSearchCV(Pipeline([("prep",clone(prep)),("clf",DecisionTreeClassifier(class_weight="balanced",random_state=RANDOM_STATE))]),{"clf__max_depth":[4,6,8,12],"clf__min_samples_leaf":[20,50,100]},n_iter=min(N_ITER,12),scoring="f1_macro",cv=cv,random_state=RANDOM_STATE,n_jobs=1,refit=False)
dt_grid.fit(X_sub,y_sub)
print("Decision Tree:",dt_grid.best_params_,dt_grid.best_score_)

In [ ]:
fitted["Decision Tree"]=Pipeline([("prep",clone(prep)),("clf",DecisionTreeClassifier(class_weight="balanced",random_state=RANDOM_STATE))]).set_params(**dt_grid.best_params_).fit(X_tr,y_tr)

In [ ]:
knn_grid=RandomizedSearchCV(Pipeline([("prep",clone(prep)),("clf",KNeighborsClassifier(n_jobs=-1))]),{"clf__n_neighbors":[5,15,31,51],"clf__weights":["uniform","distance"]},n_iter=min(N_ITER,8),scoring="f1_macro",cv=cv,random_state=RANDOM_STATE,n_jobs=1,refit=False)
knn_grid.fit(X_sub,y_sub)
print("KNN:",knn_grid.best_params_,knn_grid.best_score_)

In [ ]:
fitted["KNN"]=Pipeline([("prep",clone(prep)),("clf",KNeighborsClassifier(n_jobs=-1))]).set_params(**knn_grid.best_params_).fit(X_tr,y_tr)

In [ ]:
svm_grid=RandomizedSearchCV(Pipeline([("prep",clone(prep)),("clf",CalibratedClassifierCV(LinearSVC(class_weight="balanced",dual="auto",max_iter=2000),cv=3))]),{"clf__estimator__C":[0.01,0.1,1]},n_iter=min(N_ITER,3),scoring="f1_macro",cv=cv,random_state=RANDOM_STATE,n_jobs=1,refit=False)
svm_grid.fit(X_sub,y_sub)
print("Linear SVM:",svm_grid.best_params_,svm_grid.best_score_)

In [ ]:
fitted["Linear SVM"]=Pipeline([("prep",clone(prep)),("clf",CalibratedClassifierCV(LinearSVC(class_weight="balanced",dual="auto",max_iter=2000),cv=3))]).set_params(**svm_grid.best_params_).fit(X_tr,y_tr)
print(list(fitted))

In [ ]:
models_used=pd.DataFrame([
    ("Baseline 1","Majority class","Always predicts success"),
    ("Baseline 2","Previous-run outcome","Repeats the last run's result"),
    ("Model 1","Logistic Regression","Linear model, easy to explain"),
    ("Model 2","Decision Tree","Simple if/else rules"),
    ("Model 3","KNN","Looks at the most similar past runs"),
    ("Model 4","Linear SVM","Linear boundary, probabilities calibrated"),
    ("Model 5","Random Forest","Many trees voting (bagging)"),
    ("Model 6","Hist Gradient Boosting","Trees built one after another (boosting)")],
    columns=["Role","Algorithm","In one line"])
print(models_used.to_string(index=False))

In [ ]:
val_probs={name:pipe.predict_proba(X_va)[:,1] for name,pipe in fitted.items()}
thresholds={}
for name,p in val_probs.items():
    grid=np.linspace(0.05,0.95,91)
    scores=[f1_score(y_va,(p>=t).astype(int),average="macro") for t in grid]
    thresholds[name]=float(grid[int(np.argmax(scores))])
print(thresholds)

In [ ]:
model_rows=[]
for name,p in val_probs.items():
    pred=(p>=thresholds[name]).astype(int)
    model_rows.append({"Model":name,"macro_F1":f1_score(y_va,pred,average="macro"),"fail_F1":f1_score(y_va,pred,pos_label=1),"fail_prec":precision_score(y_va,pred,zero_division=0),"fail_rec":recall_score(y_va,pred),"ROC_AUC":roc_auc_score(y_va,p),"PR_AUC":average_precision_score(y_va,p),"threshold":thresholds[name]})
model_df=pd.DataFrame(model_rows).set_index("Model").round(3)
print(model_df)

In [ ]:
val_table=pd.concat([base_df,model_df])
print(val_table)

In [ ]:
best_name=model_df['macro_F1'].idxmax()
final=fitted[best_name]
thr=thresholds[best_name]
print("Selected model:",best_name,"| threshold",thr)

In [ ]:
p_te=final.predict_proba(X_te)[:,1]
print(p_te.shape)

In [ ]:
p_dummy_te=Pipeline([("prep",clone(prep)),("clf",DummyClassifier(strategy="most_frequent"))]).fit(X_tr,y_tr).predict_proba(X_te)[:,1]
rule_te=test['wf_prev'].fillna(test['repo_prev']).fillna(0).values

In [ ]:
test_rows=[]
for name,proba,t in [("Baseline: majority class",p_dummy_te,0.5),("Baseline: previous-run outcome",rule_te,0.5),(f"{best_name} (tuned)",p_te,thr)]:
    pred=(proba>=t).astype(int)
    test_rows.append({"Model":name,"macro_F1":f1_score(y_te,pred,average="macro"),"fail_F1":f1_score(y_te,pred,pos_label=1),"fail_prec":precision_score(y_te,pred,zero_division=0),"fail_rec":recall_score(y_te,pred),"ROC_AUC":roc_auc_score(y_te,proba),"PR_AUC":average_precision_score(y_te,proba)})
test_table=pd.DataFrame(test_rows).set_index("Model").round(3)
print(test_table)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay,RocCurveDisplay,PrecisionRecallDisplay
ConfusionMatrixDisplay.from_predictions(y_te,(p_te>=thr).astype(int),display_labels=["success","failure"],colorbar=False)
plt.title("Confusion matrix (test)")
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_te,p_te)
plt.title("ROC curve (test)")
plt.show()

In [ ]:
PrecisionRecallDisplay.from_predictions(y_te,p_te)
plt.title("Precision-recall curve (test)")
plt.show()

In [ ]:
fit_rows=[]
for name,X,y in [("train",X_tr,y_tr),("validation",X_va,y_va),("test",X_te,y_te)]:
    p=final.predict_proba(X)[:,1]
    pred=(p>=thr).astype(int)
    fit_rows.append({"split":name,"macro_F1":f1_score(y,pred,average="macro"),"fail_F1":f1_score(y,pred,pos_label=1),"ROC_AUC":roc_auc_score(y,p),"PR_AUC":average_precision_score(y,p)})
fit_check=pd.DataFrame(fit_rows).set_index("split").round(3)
print(fit_check)

In [ ]:
from sklearn.inspection import permutation_importance
sample=val.sample(min(8000,len(val)),random_state=RANDOM_STATE)
imp=permutation_importance(final,sample[FEATURES],sample['label'],scoring="f1_macro",n_repeats=3,random_state=RANDOM_STATE,n_jobs=1)
imp_s=pd.Series(imp.importances_mean,index=FEATURES).sort_values()
print(imp_s.tail(12))

In [ ]:
plt.figure(figsize=(7,5))
imp_s.tail(12).plot.barh(color="#00798c")
plt.title("Permutation importance (validation, drop in macro-F1)")
plt.tight_layout()
plt.show()

In [ ]:
p_va_final=val_probs[best_name]
low_cut=min(float(np.quantile(p_va_final[y_va.values==1],0.10)),thr)
risk_bands={"low_cut":low_cut,"high_cut":thr}
print(risk_bands)

In [ ]:
def band(p):
    return np.where(p>=risk_bands["high_cut"],"High",np.where(p>=risk_bands["low_cut"],"Medium","Low"))
rows=[]
for split,p,y in [("validation",p_va_final,y_va),("test",p_te,y_te)]:
    b=band(p)
    for lvl in ["Low","Medium","High"]:
        mask=b==lvl
        rows.append({"split":split,"risk level":lvl,"runs":int(mask.sum()),"share of runs":round(mask.mean(),3),"actual failure rate":round(float(y.values[mask].mean()),3) if mask.any() else np.nan})
risk_table=pd.DataFrame(rows)
print(risk_table.to_string(index=False))

In [ ]:
import joblib
te_band=band(p_te)
risk_rates={lvl:(float(y_te.values[te_band==lvl].mean()) if (te_band==lvl).any() else 0.0) for lvl in ["Low","Medium","High"]}
risk_rates["overall"]=float(y_te.mean())
print(risk_rates)
bundle={"pipeline":final,"threshold":thr,"features":FEATURES,"log_cols":LOG_COLS,"num_cols":NUM_COLS,"cat_cols":CAT_COLS,"model_name":best_name,"risk_bands":risk_bands,"risk_rates":risk_rates,"test_metrics":test_table.loc[f"{best_name} (tuned)"].to_dict(),"feature_medians":X_tr[LOG_COLS+NUM_COLS].median().to_dict()}
joblib.dump(bundle,f"{MODEL_DIR}/ci_failure_model.joblib")
print("saved",f"{MODEL_DIR}/ci_failure_model.joblib")

In [ ]:
from importlib.metadata import version
with open(f"{BASE}/requirements.txt","w") as f:
    for pkg in ["numpy","pandas","scikit-learn","matplotlib","seaborn","joblib","streamlit","flask","gradio"]:
        try: f.write(f"{pkg}=={version(pkg)}\n")
        except Exception: pass
print(open(f"{BASE}/requirements.txt").read())

In [ ]:
data.drop(columns=['wf_key']).to_csv(f"{DATA_DIR}/cleaned_runs.csv.gz",index=False)
print("saved cleaned data")